# 07 — Vectorless RAG, and Typesense

Two remaining topics from the tutor's repo, both of which challenge assumptions the
earlier notebooks quietly built in.

**Part A — Vectorless RAG** questions the biggest assumption of all: that retrieval
must work by *similarity*. It uses **no embeddings, no vector database, and no
chunking** — an LLM reads a table of contents and reasons about where to look.

**Part B — Typesense** questions where the index should *live*. ChromaDB and FAISS
run inside your Python process; Typesense is a search **server** other applications
can share, with keyword search built in.

---
# Part A — Vectorless RAG (the PageIndex approach)

## The mechanism

Instead of embedding everything, organise documents into a **tree** and let the LLM
navigate it the way a person uses a table of contents:

```
root
├── attention_is_all_you_need.pdf
│    ├── [0] "Attention Is All You Need — An Overview..."
│    ├── [1] "Scaled Dot-Product Attention..."
│    └── [2] "Why Self-Attention..."
├── embedding_models_report.pdf
│    └── ...
└── vector_databases.xlsx
     └── ...
```

Retrieval becomes two reasoning steps:

1. **Which documents are worth opening?** — the LLM sees file names plus a short
   preview of each section.
2. **Which sections inside them?** — the LLM sees previews from just those files and
   picks what to read in full.

Then the selected sections answer the question, exactly as in normal RAG.

## Why bother

| | Similarity search | Vectorless (tree search) |
|---|---|---|
| Index | embeddings in a vector DB | a table of contents |
| Chunking | required, and needs tuning | none — sections stay whole |
| Retrieval | cosine distance | LLM reasoning |
| Cost per query | ~free vector lookup | 2–3 LLM calls |
| Scales to | millions of chunks | tens/hundreds of documents |
| Explains itself | a distance number | "I opened X because…" |

The real advantage is on questions where **no single passage is "similar" to the
query** — comparisons and multi-hop reasoning. "Which of these databases is managed
and which is open source?" isn't semantically close to any one row, but a model
reading a table of contents can still work out where to look.

This is a local port of the repo's PageIndex notebook — same mechanism, running on
Groq with no external service or extra API key.

In [1]:
import sys
sys.path.append("..")

from src.data_loader import load_all_documents
from src.vectorless_rag import VectorlessRAG

documents = load_all_documents("../data")
vectorless = VectorlessRAG(documents)

Found 3 PDF files
  loaded attention_is_all_you_need.pdf: 3 pages
  loaded embedding_models_report.pdf: 3 pages
  loaded rag_systems_overview.pdf: 2 pages
Found 2 text files
  loaded machine_learning.txt
  loaded python_intro.txt
Found 1 CSV files
  loaded ai_models.csv: 4 rows
Found 1 JSON files
  loaded rag_faq.json: 4 entries
Found 1 Excel files
  loaded vector_databases.xlsx: 4 rows
Found 1 SQLite databases
  loaded llm_catalog.db: 4 rows from 1 table(s)
Found 1 Word files
  loaded retrieval_evaluation.docx
Total documents loaded: 27


Vectorless RAG LLM initialized: openai/gpt-oss-120b
Tree: 10 documents, 27 sections, 0 embeddings


Note what that printed: **0 embeddings**. There is no index to build — startup is
just grouping documents by source file.

### The entire "index" is this text

Everything the retriever knows is the table of contents below. That's it.

In [2]:
toc = vectorless.table_of_contents()
print("\n".join(toc.splitlines()[:16]))
print("...")
print(f"\n({len(toc)} characters total — this whole string goes into the "
      f"navigation prompt)")

DOCUMENT: attention_is_all_you_need.pdf
  [0] Attention Is All You Need - An Overview The Transformer is a neural network architecture based entirely on attention mechanisms, dispensing with recurrence and...
  [1] Scaled Dot-Product Attention The particular attention used in the Transformer is called Scaled Dot-Product Attention. The input consists of queries and keys of...
  [2] Why Self-Attention Self-attention, sometimes called intra-attention, is an attention mechanism relating different positions of a single sequence in order to com...
DOCUMENT: embedding_models_report.pdf
  [0] Text Embedding Models - Technical Report Text embedding models convert natural language into dense numerical vectors that capture semantic meaning. Similar text...
  [1] Hard Negative Mining Techniques Hard negative mining is a technique for improving embedding model quality. Instead of sampling random negatives, hard negatives...
  [2] Evaluation and Dimensionality Embedding models are evaluated on bench

That last number is also the approach's ceiling: the table of contents must fit in
the prompt. Fine for tens or hundreds of documents; not a substitute for a vector
index over millions.

### A straightforward question

In [3]:
result = vectorless.print_query("What is the attention mechanism?")

Q: What is the attention mechanism?


   step 1 · opened documents: ['attention_is_all_you_need.pdf', 'machine_learning.txt']


   step 2 · selected sections: ['attention_is_all_you_need.pdf#0', 'attention_is_all_you_need.pdf#2']


A: The attention mechanism maps a **query** together with a set of **key‑value pairs** to an output vector.  
It does this by computing a compatibility score between the query and each key, turning those scores into weights (often via softmax), and then taking a weighted sum of the corresponding values. The result is the attention output. (See [attention_is_all_you_need.pdf #0])



Watch the two navigation steps in that output — the model opened the right file,
then narrowed to the right sections, using nothing but previews.

### The question where this approach earns its keep

A **comparison across two rows** of a spreadsheet. Semantically, no single row is
"similar" to this question — it asks about a relationship *between* rows.

In [4]:
result = vectorless.print_query(
    "Which vector database is a fully managed cloud service, and which is open source?"
)

Q: Which vector database is a fully managed cloud service, and which is open source?


   step 1 · opened documents: ['vector_databases.xlsx']


   step 2 · selected sections: ['vector_databases.xlsx#2', 'vector_databases.xlsx#0']


A: - **Fully managed cloud service:** Pinecone  
- **Open‑source:** ChromaDB (Apache‑2.0)



### Same question through similarity search, for contrast

In [5]:
from src.vector_store import FaissVectorStore
from src.search import RAGSearch

store = FaissVectorStore(persist_directory="../faiss_store_nb07")
if store.exists():
    store.load()
else:
    store.build_from_documents(documents)

rag = RAGSearch(store)
out = rag.search_and_summarize(
    "Which vector database is a fully managed cloud service, and which is open source?",
    top_k=3,
)
print("SIMILARITY SEARCH")
print("  chunks retrieved from:", [s["source_file"] for s in out["sources"]])
print("  answer:", out["answer"][:400])

Loading embedding model for vector store: all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading embedding model: all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded. Embedding dimension: 384
Split 27 documents into 27 chunks


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape (27, 384)
Vector store now holds 27 vectors
Saved FAISS index + metadata to '../faiss_store_nb07'


Groq LLM initialized: openai/gpt-oss-120b


SIMILARITY SEARCH
  chunks retrieved from: ['vector_databases.xlsx', 'vector_databases.xlsx', 'vector_databases.xlsx']
  answer: - **Fully managed cloud service:** Pinecone  
- **Open‑source:** Weaviate (FAISS is also open source)


Both approaches can handle this one — our corpus is small enough that `top_k=3`
happens to pull in enough rows. The difference shows up as corpora grow: similarity
search must get all the relevant rows into the top-k *by distance alone*, while tree
search deliberately selects them.

**When to reach for vectorless RAG:** small, well-structured corpora (reports,
manuals, contracts, wikis); questions needing comparison or synthesis; cases where
you must explain *why* a document was consulted. **When not to:** large corpora,
high query volume, or tight latency budgets — the LLM calls are the cost.

---
# Part B — Typesense

## Where the index lives

| | ChromaDB / FAISS | Typesense |
|---|---|---|
| Form | a library in your process | a server you connect to |
| Index location | a local file | inside the server |
| Concurrent clients | one process | many |
| Search types | vector | **vector + keyword + filters** |

The last row matters most. Notebook 04 (Part C) showed that sparse keyword search
beats dense retrieval on exact identifiers — error codes, part numbers, ticket IDs.
Typesense does both in one engine, which makes hybrid retrieval practical instead of
requiring two systems.

## This needs a server running

Typesense isn't embedded. Nothing works without a server:

```bash
docker run -p 8108:8108 -v "$(pwd)/typesense-data:/data" \
  typesense/typesense:27.1 \
  --data-dir /data --api-key=devkey --enable-cors
```

> **⚠️ Honest caveat:** unlike every other module in this project, this one has
> **not been run end-to-end**. Docker wasn't available in the environment where
> this was built, so only the API usage was verified against the installed
> `typesense` 2.0 client. Treat it as a reviewed starting point and expect to
> adjust details on first run.

`ping()` exists so you find out immediately whether a server is reachable, rather
than hitting a confusing connection error four calls later.

In [6]:
from src.typesense_store import TypesenseVectorStore

ts = TypesenseVectorStore(collection_name="rag_chunks")
server_up = ts.ping()

Loading embedding model for Typesense store: all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Typesense server NOT reachable (ConnectError). Start one with:
  docker run -p 8108:8108 -v "$(pwd)/typesense-data:/data" \
    typesense/typesense:27.1 --data-dir /data --api-key=devkey --enable-cors


If that printed the Docker command, there's no server — which is the expected
result here and why the cells below are guarded. Start a server and re-run to
exercise them.

In [7]:
if server_up:
    ts.create_collection(drop_existing=True)
    ts.build_from_documents(documents)

    print("\nVECTOR SEARCH")
    for hit in ts.query("What is the attention mechanism?", top_k=3):
        print(f"  {hit['metadata']['source_file']}  distance={hit['distance']:.4f}")

    print("\nKEYWORD SEARCH (the half FAISS and Chroma cannot do)")
    for hit in ts.keyword_query("multi-head attention", top_k=3):
        print(f"  {hit['metadata']['source_file']}  text_match={hit['text_match']}")

    print("\nFILTERED VECTOR SEARCH (metadata as a facet)")
    for hit in ts.query("attention", top_k=3, filter_by="file_type:=pdf"):
        print(f"  {hit['metadata']['source_file']} ({hit['metadata']['file_type']})")
else:
    print("Skipped — no Typesense server. Start one with the Docker command above.")

Skipped — no Typesense server. Start one with the Docker command above.


### The schema difference worth knowing

Typesense requires the vector width **declared up front**:

```python
{"name": "embedding", "type": "float[]", "num_dim": 384}
```

FAISS infers dimension from the first insert; Typesense will not. And schemas are
fixed once created — changing one means dropping and rebuilding the collection,
which is what `drop_existing=True` is for.

That strictness buys you the `facet: True` fields, which is what makes
`filter_by="file_type:=pdf"` possible — narrowing a vector search by metadata,
something a bare FAISS index cannot do at all.

---
## Choosing between all four stores

| Store | Reach for it when |
|---|---|
| **ChromaDB** | prototyping, notebooks, zero setup |
| **FAISS** | one process, maximum speed, file-based persistence |
| **Typesense** | several apps share an index; you want hybrid search and filters |
| **Vectorless** | small structured corpus; comparison/synthesis questions; explainability |

All four are interchangeable behind the same `query(text, top_k)` interface used
throughout this project — which is why `RAGSearch` and the evaluation module work
with any of them unchanged. That interface discipline is what makes swapping stores
a one-line change instead of a rewrite.